# 分散 Self-Play — Kaggle で収集と学習の両方

収集だけでなく PPO 更新と評価も Kaggle 側で行い、次世代のモデルまで作る。
手元の PC は zip の受け渡しだけになり、CPU をほとんど使わない。

入力: `run.json` / `model_v<N>.json` / `trainer_v<N>.pt` / `history.jsonl`
出力: `/kaggle/working/result.zip`(次世代モデル・学習状態・更新後の run.json・履歴・経験データ)


## 1. 展開と世代の確認


In [ ]:
!python -V
!nproc


In [ ]:
import glob, json, os, shutil, zipfile

EXPECTED_GEN = -1   # push_kaggle.py が実際の世代に書き換える
EXPECTED_RUN_ID = ''   # push_kaggle.py が実際の run 名に書き換える

# まず入力に何が来ているかを必ず出す。Kaggle は Dataset にアップロードした zip を
# 展開して置くことがあるので、zip のままでも展開済みでも動くようにする。
print('--- /kaggle/input ---')
for p in sorted(glob.glob('/kaggle/input/**', recursive=True))[:60]:
    print(' ', p)

os.makedirs('/kaggle/temp/ptcg', exist_ok=True)

def _place(zip_glob, marker_glob, marker_depth, dest):
    """zip があれば展開、無ければ展開済みの場所を marker から探してコピー。"""
    z = glob.glob(zip_glob, recursive=True)
    if z:
        print('zip から展開:', z[0])
        zipfile.ZipFile(z[0]).extractall(dest)
        return
    m = glob.glob(marker_glob, recursive=True)
    if not m:
        raise SystemExit(f'入力が見つからない: {zip_glob} も {marker_glob} も無い。'
                         'Dataset が Notebook に添付されているか確認する。')
    root = m[0]
    for _ in range(marker_depth):
        root = os.path.dirname(root)
    print('展開済みをコピー:', root)
    shutil.copytree(root, dest, dirs_exist_ok=True)

_place('/kaggle/input/**/ptcg_repo.zip',
       '/kaggle/input/**/sample_submission/cg/libcg.so', 3, '/kaggle/temp/ptcg')
# run zip は中に run/ を含むので展開先は /kaggle/temp。展開済みの場合は run/ 自体をコピー。
if glob.glob('/kaggle/input/**/run_*.zip', recursive=True):
    _place('/kaggle/input/**/run_*.zip', '', 0, '/kaggle/temp')
else:
    _place('/nonexistent', '/kaggle/input/**/run/run.json', 1, '/kaggle/temp/run')


# 必要なものが本当に来ているかを名指しで確認する(Dataset の版が古いと欠ける)
need = ['kaggle_replays/rl/distributed/worker.py',
        'kaggle_replays/rl/collect_parallel.py',
        'sample_submission/cg/libcg.so',
        'kaggle_replays/meta_analysis/archetype_decks/dragapult_ex/01.csv']
missing = [n for n in need if not os.path.exists('/kaggle/temp/ptcg/' + n)]
print('\n--- 必要ファイルの確認 ---')
for n in need:
    print(('  OK  ' if n not in missing else '  なし '), n)
if missing:
    raise SystemExit('Dataset の中身が足りない(古い版が添付された可能性)。'
                     f'欠けているもの: {missing}')


# Dataset の新バージョンが反映される前に実行されると、1つ前の世代の run.json が
# 添付される。ファイルは揃っているので存在チェックでは気づけない。ここで世代を
# 突き合わせて即座に落とす(3分かけて無駄な経験を集めてしまうのを防ぐ)。
run_cfg = json.load(open('/kaggle/temp/run/run.json', encoding='utf-8'))
print('run.json:', run_cfg['run_id'], 'v%d' % run_cfg['generation'],
      '/ 期待:', EXPECTED_RUN_ID, 'v%d' % EXPECTED_GEN)
if EXPECTED_GEN >= 0 and run_cfg['generation'] != EXPECTED_GEN:
    raise SystemExit(f"Dataset が古い: run.json は v{run_cfg['generation']} だが "
                     f"v{EXPECTED_GEN} を期待。新バージョンがまだ反映されていない。")

# run を複数並行で回すときの取り違え。Dataset は run ごとに分けてあるが、
# 添付先を間違えると別の run のモデルで学習してしまう。世代番号だけでは見抜けない。
if EXPECTED_RUN_ID and run_cfg['run_id'] != EXPECTED_RUN_ID:
    raise SystemExit(f"別の run の Dataset が添付されている: run.json は "
                     f"{run_cfg['run_id']} だが {EXPECTED_RUN_ID} を期待。")

!ls /kaggle/temp/run /kaggle/temp/run/models


## 2. 収集 → PPO更新 → 評価

worker.py で試合を集め、その場で learner.py が次世代モデルを作る。
シャードを持ち帰る必要がないので、往復が1回で済む。


In [ ]:
import subprocess, sys, os

WORKERS = 4            # push_kaggle.py が書き換える
START_METHOD = 'spawn' # push_kaggle.py が書き換える

os.chdir('/kaggle/temp/ptcg/kaggle_replays/rl/distributed')

def run(*args):
    """失敗したらセルごと止める。

    ノートブックの `!python ...` は終了コードを見ないので、worker や learner が
    落ちてもセルは成功扱いになり、ずっと後の梱包段階で初めて異常が出る。
    原因から遠い場所でエラーになって診断しづらいので、ここで落とす。
    """
    print('$', ' '.join(args), flush=True)
    r = subprocess.run([sys.executable] + list(args), text=True,
                       capture_output=True, encoding='utf-8', errors='replace')
    print(r.stdout, flush=True)
    if r.returncode != 0:
        print(r.stderr[-4000:], flush=True)
        raise SystemExit(f'{args[0]} が失敗 (exit {r.returncode})')

run('worker.py', '--run-dir', '/kaggle/temp/run', '--worker-id', 'kaggle',
    '--workers', str(WORKERS), '--start-method', START_METHOD)


In [ ]:
EVAL_EVERY = 4   # 何世代に1回評価するか

# 評価は学習には一切使われず、経過を見るためだけのもの。1世代あたり600試合=全体の
# 56% を消費するわりに、400試合の誤差は±0.05 で世代ごとの比較には使えない。
# 傾向を追うだけなら間引いて十分なので、その世代だけ回す。
produced = EXPECTED_GEN + 1
do_eval = (produced % EVAL_EVERY == 0)
print(f'v{produced} の評価: ' + ('実施' if do_eval else '省略(学習には影響しない)'))

run('learner.py', '--run-dir', '/kaggle/temp/run',
    '--eval-games', '200' if do_eval else '0',
    '--eval-pool-games', '400' if do_eval else '0',
    '--eval-workers', str(WORKERS), '--start-method', START_METHOD, '--no-inbox')


## 3. 結果をまとめて出力


In [ ]:
import json, shutil, zipfile, glob, os

run_cfg = json.load(open('/kaggle/temp/run/run.json', encoding='utf-8'))
gen = run_cfg['generation']          # learner が +1 済み
print('更新後の世代:', gen)

out = '/kaggle/working/result.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('/kaggle/temp/run/run.json', 'run.json')
    z.write('/kaggle/temp/run/history.jsonl', 'history.jsonl')
    z.write(f'/kaggle/temp/run/models/model_v{gen}.json', f'models/model_v{gen}.json')
    st = f'/kaggle/temp/run/state/trainer_v{gen}.pt'
    if os.path.exists(st):
        z.write(st, f'state/trainer_v{gen}.pt')
    # 使い終わった経験データも持ち帰る(手元に記録を残すため)
    for p in glob.glob(f'/kaggle/temp/run/shards/consumed/v{gen-1}/*.npz'):
        z.write(p, f'shards/consumed/v{gen-1}/' + os.path.basename(p))
print(out, os.path.getsize(out) / 1e6, 'MB')
!ls -lh /kaggle/working/
